# G17 — BrEaST failure breakdown by clinical attributes (CPU, seconds)
Inputs: **G17_external_BrEaST** output (per-scan `ext_*.csv`) + **Breast-Lesions-USG** (clinical xlsx). Groups Dice of the 8-model ensemble by every categorical column (scanner, pathology, BI-RADS, shape, margin, …) and correlates it with every numeric one.

In [ ]:
!pip -q install openpyxl

In [ ]:
import glob, os, re, numpy as np, pandas as pd
from scipy.stats import spearmanr, kruskal
f = sorted(glob.glob('/kaggle/input/**/ext_ENS_upp_all.csv', recursive=True)) or sorted(glob.glob('/kaggle/input/**/ext_*.csv', recursive=True))
d = pd.read_csv(f[0]); print('per-scan file:', os.path.basename(f[0]), d.shape)
x = sorted(glob.glob('/kaggle/input/**/*clinical*.xlsx', recursive=True))[0]
c = pd.read_excel(x); print('clinical:', c.shape); print(c.columns.tolist()); print(c.head(3).T)
key = None
for col in c.columns:
    s = c[col].astype(str)
    if s.str.contains(r'case', case=False).mean() > 0.5: key = col; break
print('join column:', key)
c['image_id'] = c[key].astype(str).str.replace(r'\.png$', '', regex=True).str.extract(r'(case\d+)', expand=False)
m = d.merge(c, on='image_id', how='left'); print('matched', m[key].notna().sum(), 'of', len(m))
m['dice100'] = 100 * m['dice']; m['fail'] = m['dice'] < 0.5

In [ ]:
print('=== OVERALL', round(m.dice100.mean(), 2), 'fail%', round(100 * m.fail.mean(), 1))
for col in c.columns:
    if col in (key, 'image_id'): continue
    s = m[col]
    if s.dtype.kind in 'if' and s.nunique() > 10:
        ok = s.notna()
        if ok.sum() > 20:
            r, p = spearmanr(s[ok], m.dice[ok]); print(f'\n[numeric] {col}: Spearman rho={r:.2f} p={p:.3g}')
            q = pd.qcut(s[ok], 3, duplicates='drop'); print(m[ok].groupby(q, observed=True).agg(n=('dice100', 'size'), dice=('dice100', 'mean'), fail_pct=('fail', 'mean')).assign(fail_pct=lambda t: 100 * t.fail_pct).round(1).to_string())
    elif 1 < s.nunique() <= 12:
        g = m.groupby(s.astype(str)).agg(n=('dice100', 'size'), dice=('dice100', 'mean'), fail_pct=('fail', 'mean')).assign(fail_pct=lambda t: 100 * t.fail_pct)
        g = g[g.n >= 3].sort_values('dice')
        groups = [m.dice[s.astype(str) == k] for k in g.index if (s.astype(str) == k).sum() >= 3]
        p = kruskal(*groups).pvalue if len(groups) > 1 else float('nan')
        print(f'\n[categorical] {col}  (Kruskal p={p:.3g})'); print(g.round(1).to_string())
m.sort_values('dice').head(30)[['image_id', 'dice100'] + [k for k in c.columns if k != key][:12]].to_csv('worst30_breast.csv', index=False)
print('\nworst 30 written')